# Analysis Results

In [4]:
import pandas as pd
import re
from pathlib import Path

In [9]:
RESULTS_FOLDER = Path("/home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results")

CLEAN_DATA_FOLDER = Path("/home/silvia/Documents/GraduationProject/research-questions/RQ1_new/analysis/clean_data")

In [ ]:
MODELS = ["model1", "model3", "model3-w2", "model3-w3", "model3-w4"]

NAME_RE = re.compile(
    r"^(?P<model>model\d(?:-w\d)?)"
    r"(?:_(?P<arm>sib_cfs|sib_res|phys_cfs|phys_res))?"
    r"(?:_(?P<workload>ptrchase))?"
    r"_round(?P<round>\d+)$"
)

for model in MODELS:
    by_workload = {"matmul": [], "ptrchase": []}

    for exp_dir in RESULTS_FOLDER.glob(f"{model}_*round*"):
        m = NAME_RE.match(exp_dir.name)
        if not m or m.group("model") != model:
            continue
        arm = m.group("arm") or "base"
        workload = m.group("workload") or "matmul"
        round_n = int(m.group("round"))

        for jobs_csv in exp_dir.glob("*/*/jobs.csv"):
            scale = jobs_csv.parent.parent.name
            u = jobs_csv.parent.name
            df = pd.read_csv(jobs_csv, comment="#")
            df["model"] = model
            df["arm"] = arm
            df["round"] = round_n
            df["scale"] = scale
            df["U"] = u
            by_workload[workload].append(df)

    for workload, rows in by_workload.items():
        if not rows:
            print(f"[skip] no {workload} result dirs found for {model}")
            continue
        joined_df = pd.concat(rows, ignore_index=True)
        out_path = CLEAN_DATA_FOLDER / f"{model}_{workload}_joined.csv"
        joined_df.to_csv(out_path, index=False)
        print(f"[ok] {model} {workload}: {len(joined_df)} rows -> {out_path}")


[ok] model1 matmul: 400000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model1_matmul_joined.csv
[ok] model1 ptrchase: 395000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model1_ptrchase_joined.csv
[ok] model3 matmul: 410000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model3_matmul_joined.csv
[ok] model3 ptrchase: 395000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model3_ptrchase_joined.csv
[ok] model3-w2 matmul: 410000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model3-w2_matmul_joined.csv
[ok] model3-w2 ptrchase: 395000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model3-w2_ptrchase_joined.csv
[ok] model3-w3 matmul: 400000 rows -> /home/silvia/Documents/GraduationProject/research-questions/RQ1_new/results/model3-w3_matmul_joined.csv
[ok] model3-w3 ptr